# Measure chunk latency

Run this notebook from its directory on the target SSD. It uses the upstream native profiler to measure seven blocks and writes `block_estimates.csv` here.

After measurement, run [01_02_chunk_latency.ipynb](01_02_chunk_latency.ipynb) to fit and evaluate the latency models.

If the upstream profiler is absent, initialize it from the repository root with `git submodule update --init --recursive vlm-flash`. Acquisition requires its native build dependencies and this notebook directory on the target SSD.

In [1]:
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd

DIRECTORY = Path.cwd()
SUBMODULE = DIRECTORY / "../../vlm-flash"
SIZES_KIB = np.r_[np.arange(1, 257), np.arange(260, 769, 4)]
BLOCKS, ITERS, WARMUP, THREADS, BLOB_MIB = 7, 10, 3, 6, 128

## 1. Prepare the native reader

Load the upstream profiler and require direct I/O for each read.

In [ ]:
sys.path[:0] = [str((SUBMODULE / p).resolve()) for p in ("scripts", "src")]
import profile_flash as profiler

native_reader = profiler.native()
if native_reader is None:
    raise RuntimeError(profiler.unavailable_reason())

class DirectOnly:
    def read_rows(self, *args, **kwargs):
        result = native_reader.read_rows(*args, **kwargs)
        if not result[3]:
            raise RuntimeError("O_DIRECT unavailable; no latency profile was saved.")
        return result

reader = DirectOnly()

## 2. Measure each chunk size

Randomize size order within each block and convert batch throughput to amortized chunk cost:

$$T_b(z)=\frac{1000z}{1024\widehat\beta_b(z)}\quad\mathrm{ms/chunk}.$$

The native reader must use direct I/O.

The profiler smooths throughput with a three-point mean, then averages the tail after the first absolute slope below 5% of peak throughput per MiB. If none qualifies, it averages the last 20% of points; with fewer than three points it uses the last. Throughput uses logical bytes, excluding alignment padding. The resulting cost is amortized batch cost and does not by itself certify saturation.

In [ ]:
num_rows = BLOB_MIB * 1024 // profiler.ROW_KB
rng, records = np.random.default_rng(0), []
with profiler.exclusive("01_chunk_latency"), TemporaryDirectory(dir=DIRECTORY) as work:
    blob = str(Path(work) / "profile_blob.dat")
    profiler.prepare_blob(blob, num_rows)
    for block in range(BLOCKS):
        for order, size in enumerate(rng.permutation(SIZES_KIB)):
            size = int(size)
            runs = profiler.measure(reader, blob, num_rows, [size],
                                    ITERS, WARMUP, THREADS)[size]
            rate = profiler.saturation_throughput(runs, size)
            records.append(dict(block=block, order=order, size_kib=size,
                latency_ms=1000 * size / (1024 * rate),
                throughput_mib_s=rate, count_points=len(runs),
                max_count=runs[-1][0]))

In [ ]:
data = pd.DataFrame(records)
data.head()

## 3. Save the measurements

After inspecting the table, run this cell to replace `block_estimates.csv`.

In [ ]:
data.to_csv(DIRECTORY / "block_estimates.csv", index=False)